In [1]:
import json
import math
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from tqdm import tqdm

BACKEND_DIR = Path("..").resolve()
TEST_DIR = BACKEND_DIR / "dataset" / "test_ready"
KB_DIR = BACKEND_DIR / "knowledge_base"
REZULTATI_DIR = BACKEND_DIR / "rezultati"

URL = "http://127.0.0.1:8000/odluka"
SEED = 42
KS = [1, 2, 3]
BROJ_SLIKA_MAX = 5
BROJ_POKUSAJA = 3
PAUZA_POKUSAJ_S = 10

SLIKE_CSV = REZULTATI_DIR / "test_slike.csv"
REZULTATI_CSV = REZULTATI_DIR / "test_svi_upiti.csv"

In [2]:
with open(KB_DIR / "tekst_upiti.json", encoding="utf-8") as f:
    svi_upiti = json.load(f)
vrste = sorted(s for s in svi_upiti if s != "_meta")
assert len(vrste) == 324 and set(vrste) == {p.name for p in TEST_DIR.iterdir()}

for s in vrste:
    assert {u["stil"] for u in svi_upiti[s]["test"]} >= {"short", "detailed"}, s
    assert len(list((TEST_DIR / s).glob("*.jpg"))) >= BROJ_SLIKA_MAX, s

VARIJANTE = (
    [(f"tekst_{st}", "tekst", st, 0) for st in ("short", "detailed")]
    + [(f"slike_{n}", "slike", "", n) for n in (1, 3, 5)]
    + [(f"oba_{st}_{n}", "oba", st, n) for st in ("short", "detailed") for n in (1, 3, 5)]
)
REDOSLED_VARIJANTI = [v[0] for v in VARIJANTE]

def izvuci_slike():
    rng = random.Random(SEED)
    redovi = []
    for s in vrste:
        fajlovi = sorted(p.name for p in (TEST_DIR / s).glob("*.jpg"))
        redovi.append((s, "|".join(rng.sample(fajlovi, BROJ_SLIKA_MAX))))
    return pd.DataFrame(redovi, columns=["specie_id", "slike"])

if SLIKE_CSV.exists():
    izvucene = pd.read_csv(SLIKE_CSV, dtype=str, keep_default_na=False)
else:
    izvucene = izvuci_slike()
    izvucene.to_csv(SLIKE_CSV, index=False)
slike_po_vrsti = dict(zip(izvucene["specie_id"], izvucene["slike"]))

redovi = []
for s in vrste:
    pet = slike_po_vrsti[s].split("|")
    for varijanta, modalitet, stil, n in VARIJANTE:
        redovi.append((s, varijanta, modalitet, stil, n, "|".join(pet[:n])))
upiti = pd.DataFrame(redovi, columns=["specie_id", "varijanta", "modalitet", "stil", "n_slika", "slike"])

pd.crosstab(upiti["modalitet"], upiti["varijanta"], margins=True)[REDOSLED_VARIJANTI + ["All"]]

varijanta,tekst_short,tekst_detailed,slike_1,slike_3,slike_5,oba_short_1,oba_short_3,oba_short_5,oba_detailed_1,oba_detailed_3,oba_detailed_5,All
modalitet,,,,,,,,,,,,
oba,0,0,0,0,0,324,324,324,324,324,324,1944
slike,0,0,324,324,324,0,0,0,0,0,0,972
tekst,324,324,0,0,0,0,0,0,0,0,0,648
All,324,324,324,324,324,324,324,324,324,324,324,3564


In [3]:
def upit(red):
    s = red["specie_id"]
    data = {"opis": ""}
    if red["stil"]:
        data["opis"] = next(u["upit"] for u in svi_upiti[s]["test"] if u["stil"] == red["stil"])
    imena = red["slike"].split("|") if red["slike"] else []

    for pokusaj in range(BROJ_POKUSAJA):
        fajlovi = [("slike", (ime, open(TEST_DIR / s / ime, "rb"), "image/jpeg")) for ime in imena]
        try:
            odgovor = requests.post(URL, data=data, files=fajlovi, timeout=120)
        finally:
            for _, (_, f, _) in fajlovi:
                f.close()
        if odgovor.status_code < 500:
            break
        time.sleep(PAUZA_POKUSAJ_S)

    if odgovor.status_code != 200:
        return {"tip": "", "ishod_slika": "", "ishod_tekst": "", "predvidjene": "",
                "greska": f"{odgovor.status_code}: {odgovor.text[:200]}"}
    r = odgovor.json()
    return {"tip": r["tip"], "ishod_slika": r["grane"]["slika"], "ishod_tekst": r["grane"]["tekst"],
            "predvidjene": "|".join(r["vrste"]), "greska": ""}

In [4]:
PROBA = False

kolone = ["specie_id", "varijanta", "modalitet", "stil", "n_slika", "slike", "tip", "ishod_slika", "ishod_tekst",
          "predvidjene", "hit1", "hit2", "hit3", "greska"]

gotovo = set()
if REZULTATI_CSV.exists():
    prethodni = pd.read_csv(REZULTATI_CSV, dtype=str, keep_default_na=False)
    prethodni = prethodni.drop_duplicates(["specie_id", "varijanta"], keep="last")
    ok = prethodni[prethodni["greska"] == ""]
    gotovo = set(zip(ok["specie_id"], ok["varijanta"]))

na_redu = upiti[[(s, v) not in gotovo for s, v in zip(upiti["specie_id"], upiti["varijanta"])]]
if PROBA:
    na_redu = na_redu.groupby("varijanta").head(1)
print(f"Na redu: {len(na_redu)} upita (već gotovo: {len(gotovo)})")

for _, red in tqdm(na_redu.iterrows(), total=len(na_redu)):
    r = upit(red)
    predvidjene = r["predvidjene"].split("|") if r["predvidjene"] else []
    hit = {f"hit{k}": int(red["specie_id"] in predvidjene[:k]) for k in KS}
    novi = pd.DataFrame([{**red.to_dict(), **r, **hit}])[kolone]
    novi.to_csv(REZULTATI_CSV, mode="a", header=not REZULTATI_CSV.exists(), index=False)

print("Gotovo.")

Na redu: 0 upita (već gotovo: 3564)


0it [00:00, ?it/s]

Gotovo.


In [5]:
def wilson(h, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    p = h / n
    centar = (p + z**2 / (2 * n)) / (1 + z**2 / n)
    pola = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
    return centar - pola, centar + pola

def tabela(df, po):
    redovi = []
    if df.empty:
        return pd.DataFrame(columns=po + ["n"] + [f"Hit@{k} {s}" for k in KS for s in ("%", "95% CI")])
    for kljuc, g in df.groupby(po):
        red = dict(zip(po, kljuc if isinstance(kljuc, tuple) else [kljuc]))
        red["n"] = len(g)
        for k in KS:
            h = int(g[f"hit{k}"].sum())
            lo, hi = wilson(h, len(g))
            red[f"Hit@{k} %"] = round(100 * h / len(g), 1)
            red[f"Hit@{k} 95% CI"] = f"{100*lo:.0f}–{100*hi:.0f}"
        redovi.append(red)
    return pd.DataFrame(redovi)

r = pd.read_csv(REZULTATI_CSV, dtype=str, keep_default_na=False).drop_duplicates(["specie_id", "varijanta"], keep="last")
r["n_slika"] = r["n_slika"].astype(int)
for k in KS:
    r[f"hit{k}"] = r[f"hit{k}"].astype(int)
print("Redova sa greškom:", (r["greska"] != "").sum(), "| odrađeno:", len(r), "od", len(upiti))

po_varijanti = tabela(r, ["varijanta"]).set_index("varijanta").loc[
    [v for v in REDOSLED_VARIJANTI if v in set(r["varijanta"])]].reset_index()
po_varijanti

Redova sa greškom: 0 | odrađeno: 3564 od 3564


,varijanta,n,Hit@1 %,Hit@1 95% CI,Hit@2 %,Hit@2 95% CI,Hit@3 %,Hit@3 95% CI
0,tekst_short,324,46.0,41–51,55.6,50–61,59.6,54–65
1,tekst_detailed,324,72.8,68–77,82.4,78–86,86.7,83–90
2,slike_1,324,74.7,70–79,88.6,85–92,91.4,88–94
3,slike_3,324,80.2,76–84,91.4,88–94,96.3,94–98
4,slike_5,324,81.8,77–86,93.8,91–96,96.3,94–98
5,oba_short_1,324,84.0,80–88,94.4,91–96,96.3,94–98
6,oba_short_3,324,86.1,82–89,95.7,93–97,98.5,96–99
7,oba_short_5,324,85.5,81–89,96.3,94–98,97.8,96–99
8,oba_detailed_1,324,93.8,91–96,98.8,97–100,98.8,97–100
9,oba_detailed_3,324,94.1,91–96,98.5,96–99,99.1,97–100


In [6]:
def po_modalitetu(df, ponavljanja=2000):
    po_vrsti = df.groupby(["modalitet", "specie_id"])[[f"hit{k}" for k in KS]].mean().reset_index()
    rng = np.random.default_rng(SEED)
    redovi = []
    for modalitet in ("tekst", "slike", "oba"):
        g = po_vrsti[po_vrsti["modalitet"] == modalitet]
        if g.empty:
            continue
        M = g[[f"hit{k}" for k in KS]].to_numpy()
        boot = M[rng.integers(0, len(M), (ponavljanja, len(M)))].mean(axis=1)
        lo, hi = np.percentile(boot, [2.5, 97.5], axis=0)
        red = {"modalitet": modalitet, "n vrsta": len(M)}
        for i, k in enumerate(KS):
            red[f"Hit@{k} %"] = round(100 * M[:, i].mean(), 1)
            red[f"Hit@{k} 95% CI"] = f"{100*lo[i]:.0f}–{100*hi[i]:.0f}"
        redovi.append(red)
    return pd.DataFrame(redovi)

po_modalitetu(r)

,modalitet,n vrsta,Hit@1 %,Hit@1 95% CI,Hit@2 %,Hit@2 95% CI,Hit@3 %,Hit@3 95% CI
0,tekst,324,59.4,55–64,69.0,65–73,73.1,69–77
1,slike,324,78.9,75–82,91.3,89–94,94.7,93–96
2,oba,324,89.7,87–92,97.0,96–98,98.3,97–99


In [ ]:
po_vrsti_ukupno = r.groupby(["specie_id", "modalitet"])[[f"hit{k}" for k in KS]].mean().groupby("specie_id").mean()
M = po_vrsti_ukupno.to_numpy()
rng = np.random.default_rng(SEED)
boot = M[rng.integers(0, len(M), (2000, len(M)))].mean(axis=1)
lo, hi = np.percentile(boot, [2.5, 97.5], axis=0)
print("Upita po modalitetu:", r["modalitet"].value_counts().to_dict(), "| ukupno:", len(r))
pd.DataFrame({"": ["ukupan rezultat"],
              **{f"Hit@{k} %": [round(100 * M[:, i].mean(), 1)] for i, k in enumerate(KS)},
              **{f"Hit@{k} 95% CI": [f"{100*lo[i]:.0f}–{100*hi[i]:.0f}"] for i, k in enumerate(KS)}})

In [9]:
def mcnemar_tacan(samo_a, samo_b):
    n = samo_a + samo_b
    if n == 0:
        return 1.0
    p = 2 * sum(math.comb(n, i) for i in range(min(samo_a, samo_b) + 1)) / 2**n
    return min(1.0, p)

pivot = r.pivot(index="specie_id", columns="varijanta", values="hit1")
PAROVI = [
    ("tekst_short", "tekst_detailed"),
    ("slike_1", "slike_3"), ("slike_3", "slike_5"), ("slike_1", "slike_5"),
    ("oba_short_1", "oba_short_3"), ("oba_short_3", "oba_short_5"),
    ("oba_detailed_1", "oba_detailed_3"), ("oba_detailed_3", "oba_detailed_5"),
    ("tekst_short", "oba_short_3"), ("slike_3", "oba_short_3"),
    ("tekst_detailed", "oba_detailed_3"), ("slike_3", "oba_detailed_3"),
]
redovi = []
for a, b in PAROVI:
    if a not in pivot or b not in pivot:
        continue
    samo_a = int(((pivot[a] == 1) & (pivot[b] == 0)).sum())
    samo_b = int(((pivot[a] == 0) & (pivot[b] == 1)).sum())
    redovi.append({"A": a, "B": b, "Hit@1 A %": round(100 * pivot[a].mean(), 1), "Hit@1 B %": round(100 * pivot[b].mean(), 1),
                   "samo A tačno": samo_a, "samo B tačno": samo_b, "p (McNemar)": round(mcnemar_tacan(samo_a, samo_b), 4)})
pd.DataFrame(redovi)

,A,B,Hit@1 A %,Hit@1 B %,samo A tačno,samo B tačno,p (McNemar)
0,tekst_short,tekst_detailed,46.0,72.8,23,110,0.0000
1,slike_1,slike_3,74.7,80.2,18,36,0.0198
2,slike_3,slike_5,80.2,81.8,16,21,0.5114
3,slike_1,slike_5,74.7,81.8,27,50,0.0117
4,oba_short_1,oba_short_3,84.0,86.1,8,15,0.2100
5,oba_short_3,oba_short_5,86.1,85.5,8,6,0.7905
6,oba_detailed_1,oba_detailed_3,93.8,94.1,5,6,1.0000
7,oba_detailed_3,oba_detailed_5,94.1,94.4,1,2,1.0000
8,tekst_short,oba_short_3,46.0,86.1,2,132,0.0000
9,slike_3,oba_short_3,80.2,86.1,27,46,0.0344


In [8]:
display(pd.crosstab(r["varijanta"], r["tip"].replace("", "greška"), margins=True).loc[
    [v for v in REDOSLED_VARIJANTI if v in set(r["varijanta"])] + ["All"]])

sigurni = tabela(r[r["tip"] == "siguran"], ["varijanta"])[["varijanta", "n", "Hit@1 %", "Hit@1 95% CI"]]
sigurni = sigurni.rename(columns={"n": "n sigurnih", "Hit@1 %": "tačnost sigurnih %", "Hit@1 95% CI": "95% CI"})
sigurni["udeo sigurnih %"] = (100 * sigurni["n sigurnih"] / 324).round(1)
sigurni.set_index("varijanta").loc[[v for v in REDOSLED_VARIJANTI if v in set(sigurni["varijanta"])]].reset_index()

tip,nesiguran,no_result,siguran,All
varijanta,,,,
tekst_short,284,0,40,324
tekst_detailed,213,0,111,324
slike_1,182,12,130,324
slike_3,186,0,138,324
slike_5,185,0,139,324
oba_short_1,224,0,100,324
oba_short_3,222,0,102,324
oba_short_5,220,0,104,324
oba_detailed_1,157,0,167,324


,varijanta,n sigurnih,tačnost sigurnih %,95% CI,udeo sigurnih %
0,tekst_short,40,82.5,68–91,12.3
1,tekst_detailed,111,96.4,91–99,34.3
2,slike_1,130,98.5,95–100,40.1
3,slike_3,138,99.3,96–100,42.6
4,slike_5,139,99.3,96–100,42.9
5,oba_short_1,100,99.0,95–100,30.9
6,oba_short_3,102,99.0,95–100,31.5
7,oba_short_5,104,99.0,95–100,32.1
8,oba_detailed_1,167,99.4,97–100,51.5
9,oba_detailed_3,177,98.9,96–100,54.6


In [10]:
redovi = []
for naziv, maska in [("tekst, short", r["varijanta"] == "tekst_short"),
                     ("tekst, detailed", r["varijanta"] == "tekst_detailed"),
                     ("tekst (ukupno)", r["modalitet"] == "tekst"),
                     ("slike", r["modalitet"] == "slike"),
                     ("tekst + slike", r["modalitet"] == "oba"),
                     ("svi upiti", r["modalitet"].notna())]:
    g = r[maska]
    sg = g[g["tip"] == "siguran"]
    redovi.append({"upiti": naziv, "n upita": len(g), "sigurnih": len(sg),
                   "udeo sigurnih %": round(100 * len(sg) / len(g), 1),
                   "tačnih među sigurnim": int(sg["hit1"].sum()),
                   "tačnost sigurnih %": round(100 * sg["hit1"].mean(), 1)})
pd.DataFrame(redovi)

,upiti,n upita,sigurnih,udeo sigurnih %,tačnih među sigurnim,tačnost sigurnih %
0,"tekst, short",324,40,12.3,33,82.5
1,"tekst, detailed",324,111,34.3,107,96.4
2,tekst (ukupno),648,151,23.3,140,92.7
3,slike,972,407,41.9,403,99.0
4,tekst + slike,1944,823,42.3,815,99.0
5,svi upiti,3564,1381,38.7,1358,98.3


In [ ]:
def grafikon(t, kolona, naslov, ax):
    oznake = t[kolona].astype(str)
    sirina = 0.26
    x = np.arange(len(t))
    for i, k in enumerate(KS):
        ax.bar(x + (i - 1) * sirina, t[f"Hit@{k} %"], sirina, label=f"Hit@{k}")
    ax.set_xticks(x, oznake)
    ax.set_ylim(0, 100)
    ax.set_ylabel("HitRate (%)")
    ax.set_title(naslov)
    ax.legend(loc="lower right")

fig, osa = plt.subplots(1, 2, figsize=(15, 4.5), gridspec_kw={"width_ratios": [1, 3]})
grafikon(po_modalitetu(r), "modalitet", "Po modalitetu (isti 324 vrste)", osa[0])
grafikon(po_varijanti, "varijanta", "Po varijanti (isti 324 vrste)", osa[1])
osa[1].tick_params(axis="x", labelsize=8, rotation=45)
plt.tight_layout()
plt.savefig(REZULTATI_DIR / "test_hitrate.png", dpi=150)
plt.show()

In [ ]:
imena = pd.read_csv(BACKEND_DIR / "dataset" / "specie_id - scientific_name mapping.csv", dtype=str)
print(imena.columns.tolist())

In [ ]:
ime_po_id = dict(zip(imena.iloc[:, 0], imena.iloc[:, 1]))
def nazivi(lista):
    return " | ".join(ime_po_id.get(x, x) for x in lista.split("|")) if lista else "(nema odgovora)"

greske = r[r["hit3"] == 0].copy()
greske["prava vrsta"] = greske["specie_id"].map(ime_po_id)
greske["sistem predložio"] = greske["predvidjene"].map(nazivi)
greske[["varijanta", "tip", "prava vrsta", "sistem predložio"]].sort_values(["varijanta", "tip"])